# Détection de la maladie d'Alzheimer à partir d'IRM cérébrales
## Notebook final propre — PCD ENSI 2026

Cette version est construite à partir du **notebook final original** et conserve le code du pipeline ayant produit le run final hybride.

### Pipeline
1. Préparation de l'environnement et HippoDeep
2. Segmentation hippocampique
3. Extraction des ROI 3D
4. Chargement / préparation de ResNet34 MedicalNet
5. Entraînement du CNN 3D
6. Extraction de `cnn_prob_AD`
7. Extraction des caractéristiques anatomiques/statistiques
8. Modèle hybride BernoulliNB
9. Évaluation finale et sauvegarde des résultats





## 1. Installation et préparation de l'environnement

Cette cellule reprend l'installation HippoDeep du notebook original.


In [ ]:
!git clone https://github.com/bthyreau/hippodeep_pytorch.git /kaggle/working/hippodeep_pytorch
!pip install -q nibabel scipy pandas tqdm matplotlib

## 2. Prétraitement IRM — HippoDeep et ROI




In [ ]:
# =========================================================
# PIPELINE COMPLET :
# HippoDeep segmentation + visualisation hippocampe + ROI
# Dataset AD/CN .nii ou .nii.gz
# =========================================================

import os
import shutil
import subprocess
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import zoom
from tqdm import tqdm


In [ ]:
# =========================================================
# 1) CONFIGURATION
# =========================================================

FINAL_DATASET_ROOT = "/kaggle/input/datasets/eyamaaloul/final-sc-1y-2y-3y-no-duplicates"

CLASS_INPUTS = {
    "AD": os.path.join(FINAL_DATASET_ROOT, "AD"),
    "CN": os.path.join(FINAL_DATASET_ROOT, "CN"),
}

CLASS_LABELS = {
    "AD": 1,
    "CN": 0,
}

DATASET_NAME = "FINAL_SC_1Y_15T"

HIPPODEEP_DIR = "/kaggle/working/hippodeep_pytorch"
HIPPODEEP_SCRIPT = os.path.join(HIPPODEEP_DIR, "hippodeep.py")

OUTPUT_ROOT = f"/kaggle/working/hippodeep_roi_{DATASET_NAME}"

TARGET_SHAPE = (64, 64, 64)
MARGIN = 10
USE_ZSCORE = True

# Pour tester vite : mets 3 ou 5
# Pour traiter tout le dataset : mets None
MAX_PATIENTS = None

# Si True : ne refait pas le ROI s'il existe déjà
SKIP_IF_ROI_EXISTS = True

# Si True : génère aussi une visualisation 3D du ROI
# Si Kaggle devient lent, mets False
SAVE_3D_VIEW = True

# Nombre d'images à afficher à la fin
N_SHOW = 6

print("FINAL_DATASET_ROOT :", FINAL_DATASET_ROOT)
print("AD existe :", os.path.exists(CLASS_INPUTS["AD"]))
print("CN existe :", os.path.exists(CLASS_INPUTS["CN"]))
print("HIPPODEEP_SCRIPT existe :", os.path.exists(HIPPODEEP_SCRIPT))
print("OUTPUT_ROOT :", OUTPUT_ROOT)


In [ ]:
# =========================================================
# 2) FONCTIONS UTILITAIRES
# =========================================================

def safe_mkdir(path):
    os.makedirs(path, exist_ok=True)


safe_mkdir(OUTPUT_ROOT)

# Dossiers de visualisation
QC_ROOT = os.path.join(OUTPUT_ROOT, "qc_visualisations")
QC_ORIGINAL_DIR = os.path.join(QC_ROOT, "original_mri")
QC_MASK_DIR = os.path.join(QC_ROOT, "hippocampus_masks")
QC_OVERLAY_DIR = os.path.join(QC_ROOT, "overlay_mri_mask")
QC_ROI_DIR = os.path.join(QC_ROOT, "roi_2d")
QC_ROI_3D_DIR = os.path.join(QC_ROOT, "roi_3d")

for d in [
    QC_ROOT,
    QC_ORIGINAL_DIR,
    QC_MASK_DIR,
    QC_OVERLAY_DIR,
    QC_ROI_DIR,
    QC_ROI_3D_DIR
]:
    safe_mkdir(d)


def list_mri_files(input_root):
    files = sorted([
        f for f in os.listdir(input_root)
        if f.endswith(".nii") or f.endswith(".nii.gz")
    ])

    if MAX_PATIENTS is not None:
        files = files[:MAX_PATIENTS]

    return files


def strip_extension(filename):
    if filename.endswith(".nii.gz"):
        return filename[:-7]
    elif filename.endswith(".nii"):
        return filename[:-4]
    return filename


def load_nifti(path):
    img = nib.load(path)
    data = img.get_fdata().astype(np.float32)
    data = np.nan_to_num(data)
    return data, img


def copy_input_to_working(src_path, dst_path):
    """
    HippoDeep écrit ses sorties dans /kaggle/working.
    On copie un seul fichier à la fois pour éviter les problèmes d'espace.
    """
    if os.path.exists(dst_path):
        os.remove(dst_path)

    shutil.copy2(src_path, dst_path)


def run_hippodeep_on_patient(input_nii_working):
    """
    Lance HippoDeep sur un fichier NIfTI.
    Commande utilisée :
    python /kaggle/working/hippodeep_pytorch/hippodeep.py image.nii.gz
    """
    cmd = [
        "python",
        HIPPODEEP_SCRIPT,
        input_nii_working
    ]

    print("Commande HippoDeep :", " ".join(cmd))

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("❌ HippoDeep erreur")
        print("STDOUT :")
        print(result.stdout[-1500:])
        print("STDERR :")
        print(result.stderr[-1500:])

    return result.returncode


def expected_hippodeep_outputs(working_nii_path):
    """
    HippoDeep génère normalement :
    patient_mask_L.nii.gz
    patient_mask_R.nii.gz
    patient_brain_mask.nii.gz
    patient_hippoLR_volumes.csv
    dans /kaggle/working.
    """
    basename = os.path.basename(working_nii_path)
    patient_id = strip_extension(basename)

    mask_L = f"/kaggle/working/{patient_id}_mask_L.nii.gz"
    mask_R = f"/kaggle/working/{patient_id}_mask_R.nii.gz"
    brain_mask = f"/kaggle/working/{patient_id}_brain_mask.nii.gz"
    vol_csv = f"/kaggle/working/{patient_id}_hippoLR_volumes.csv"

    return patient_id, mask_L, mask_R, brain_mask, vol_csv


def move_hippodeep_outputs(patient_id, mask_L, mask_R, brain_mask, vol_csv, dest_dir):
    """
    Déplace les sorties HippoDeep dans un dossier patient.
    """
    patient_out_dir = os.path.join(dest_dir, patient_id)
    safe_mkdir(patient_out_dir)

    for src in [mask_L, mask_R, brain_mask, vol_csv]:
        if os.path.exists(src):
            dst = os.path.join(patient_out_dir, os.path.basename(src))

            if os.path.exists(dst):
                os.remove(dst)

            shutil.move(src, dst)

    return patient_out_dir


def get_union_mask(mask_l_path, mask_r_path):
    """
    Fusion hippocampe gauche + hippocampe droit.
    """
    mask_L, _ = load_nifti(mask_l_path)
    mask_R, _ = load_nifti(mask_r_path)

    union_mask = np.logical_or(mask_L > 0, mask_R > 0)
    return union_mask


def get_bounding_box(mask):
    coords = np.argwhere(mask)

    if coords.shape[0] == 0:
        return None

    x_min, y_min, z_min = coords.min(axis=0)
    x_max, y_max, z_max = coords.max(axis=0)

    return x_min, x_max, y_min, y_max, z_min, z_max


def expand_bounding_box(bbox, volume_shape, margin):
    x_min, x_max, y_min, y_max, z_min, z_max = bbox
    sx, sy, sz = volume_shape

    x_min = max(0, x_min - margin)
    x_max = min(sx - 1, x_max + margin)

    y_min = max(0, y_min - margin)
    y_max = min(sy - 1, y_max + margin)

    z_min = max(0, z_min - margin)
    z_max = min(sz - 1, z_max + margin)

    return x_min, x_max, y_min, y_max, z_min, z_max


def crop_volume(volume, bbox):
    x_min, x_max, y_min, y_max, z_min, z_max = bbox

    return volume[
        x_min:x_max + 1,
        y_min:y_max + 1,
        z_min:z_max + 1
    ]


def resize_volume(volume, target_shape):
    current_shape = volume.shape

    zoom_factors = [
        target_shape[0] / current_shape[0],
        target_shape[1] / current_shape[1],
        target_shape[2] / current_shape[2],
    ]

    resized = zoom(volume, zoom=zoom_factors, order=1)
    return resized.astype(np.float32)


def normalize_volume(volume, use_zscore=True):
    volume = volume.astype(np.float32)

    if use_zscore:
        mean = volume.mean()
        std = volume.std()

        if std < 1e-8:
            return np.zeros_like(volume, dtype=np.float32)

        volume = (volume - mean) / std

    else:
        vmin = volume.min()
        vmax = volume.max()

        if vmax - vmin < 1e-8:
            return np.zeros_like(volume, dtype=np.float32)

        volume = (volume - vmin) / (vmax - vmin)

    return volume.astype(np.float32)


In [ ]:
# =========================================================
# 3) FONCTIONS DE VISUALISATION
# =========================================================

def save_mri_views(volume, out_path, title="IRM originale"):
    """
    Sauvegarde 3 vues noir et blanc de l'IRM :
    sagittale, coronale, axiale.
    """
    x_mid = volume.shape[0] // 2
    y_mid = volume.shape[1] // 2
    z_mid = volume.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(np.rot90(volume[x_mid, :, :]), cmap="gray")
    axes[0].set_title("Sagittale")

    axes[1].imshow(np.rot90(volume[:, y_mid, :]), cmap="gray")
    axes[1].set_title("Coronale")

    axes[2].imshow(np.rot90(volume[:, :, z_mid]), cmap="gray")
    axes[2].set_title("Axiale")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def save_mask_views(mask, out_path, title="Masque hippocampe"):
    """
    Sauvegarde 3 vues du masque hippocampe en noir et blanc.
    """
    x_mid = mask.shape[0] // 2
    y_mid = mask.shape[1] // 2
    z_mid = mask.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(np.rot90(mask[x_mid, :, :]), cmap="gray")
    axes[0].set_title("Sagittale")

    axes[1].imshow(np.rot90(mask[:, y_mid, :]), cmap="gray")
    axes[1].set_title("Coronale")

    axes[2].imshow(np.rot90(mask[:, :, z_mid]), cmap="gray")
    axes[2].set_title("Axiale")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def save_overlay_views(volume, mask, out_path, title="IRM + hippocampe"):
    """
    Superpose le masque hippocampe sur l'IRM.
    L'IRM reste en noir et blanc, le masque est affiché en orange.
    """
    x_mid = volume.shape[0] // 2
    y_mid = volume.shape[1] // 2
    z_mid = volume.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(np.rot90(volume[x_mid, :, :]), cmap="gray")
    axes[0].imshow(np.rot90(mask[x_mid, :, :]), cmap="autumn", alpha=0.45)
    axes[0].set_title("Sagittale")

    axes[1].imshow(np.rot90(volume[:, y_mid, :]), cmap="gray")
    axes[1].imshow(np.rot90(mask[:, y_mid, :]), cmap="autumn", alpha=0.45)
    axes[1].set_title("Coronale")

    axes[2].imshow(np.rot90(volume[:, :, z_mid]), cmap="gray")
    axes[2].imshow(np.rot90(mask[:, :, z_mid]), cmap="autumn", alpha=0.45)
    axes[2].set_title("Axiale")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def save_roi_views(roi, out_path, title="ROI hippocampe"):
    """
    Sauvegarde 3 vues 2D du ROI hippocampe en noir et blanc.
    """
    x_mid = roi.shape[0] // 2
    y_mid = roi.shape[1] // 2
    z_mid = roi.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(np.rot90(roi[x_mid, :, :]), cmap="gray")
    axes[0].set_title("Sagittale")

    axes[1].imshow(np.rot90(roi[:, y_mid, :]), cmap="gray")
    axes[1].set_title("Coronale")

    axes[2].imshow(np.rot90(roi[:, :, z_mid]), cmap="gray")
    axes[2].set_title("Axiale")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def save_roi_3d_view(roi, out_path, title="ROI hippocampe 3D", threshold_percentile=85):
    """
    Visualisation 3D simple du ROI avec les voxels les plus intenses.
    Ce n'est pas une reconstruction médicale parfaite,
    mais elle permet de vérifier la forme globale du ROI.
    """
    thr = np.percentile(roi, threshold_percentile)
    coords = np.argwhere(roi > thr)

    if len(coords) == 0:
        thr = np.percentile(roi, 75)
        coords = np.argwhere(roi > thr)

    values = roi[roi > thr]

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        coords[:, 2],
        c=values,
        cmap="gray",
        s=4,
        alpha=0.7
    )

    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def show_images_from_folder(folder, n=6, title="Images"):
    files = sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".png")
    ])

    print("\n" + title)
    print("Nombre :", len(files))

    for img_path in files[:n]:
        img = plt.imread(img_path)

        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(os.path.basename(img_path))
        plt.axis("off")
        plt.show()


In [ ]:
# =========================================================
# 4) TRAITEMENT D'UN PATIENT
# =========================================================

def process_one_patient(filename, class_name, input_root, hippo_mask_dir, roi_output_dir, label):
    raw_path = os.path.join(input_root, filename)
    patient_id = strip_extension(filename)

    safe_mkdir(roi_output_dir)

    roi_output_path = os.path.join(roi_output_dir, f"{patient_id}_roi.npy")

    # chemins QC
    original_qc_path = os.path.join(QC_ORIGINAL_DIR, f"{class_name}_{patient_id}_original.png")
    mask_qc_path = os.path.join(QC_MASK_DIR, f"{class_name}_{patient_id}_mask.png")
    overlay_qc_path = os.path.join(QC_OVERLAY_DIR, f"{class_name}_{patient_id}_overlay.png")
    roi_qc_path = os.path.join(QC_ROI_DIR, f"{class_name}_{patient_id}_roi.png")
    roi_3d_qc_path = os.path.join(QC_ROI_3D_DIR, f"{class_name}_{patient_id}_roi_3d.png")

    if SKIP_IF_ROI_EXISTS and os.path.exists(roi_output_path):
        return {
            "patient_id": patient_id,
            "status": "skipped_existing_roi",
            "class_name": class_name,
            "label": label,
            "roi_path": roi_output_path,
            "qc_original_path": original_qc_path if os.path.exists(original_qc_path) else None,
            "qc_mask_path": mask_qc_path if os.path.exists(mask_qc_path) else None,
            "qc_overlay_path": overlay_qc_path if os.path.exists(overlay_qc_path) else None,
            "qc_roi_path": roi_qc_path if os.path.exists(roi_qc_path) else None,
            "qc_roi_3d_path": roi_3d_qc_path if os.path.exists(roi_3d_qc_path) else None,
        }

    # 1) Copier l'image dans /kaggle/working pour HippoDeep
    working_nii = f"/kaggle/working/{filename}"

    if os.path.exists(working_nii):
        os.remove(working_nii)

    try:
        copy_input_to_working(raw_path, working_nii)

        # 2) Lancer HippoDeep
        ret = run_hippodeep_on_patient(working_nii)

        if ret != 0:
            return {
                "patient_id": patient_id,
                "status": "hippodeep_failed",
                "class_name": class_name,
                "label": label,
                "error": f"return_code={ret}"
            }

        # 3) Sorties attendues
        _, mask_L, mask_R, brain_mask, vol_csv = expected_hippodeep_outputs(working_nii)

        if not os.path.exists(mask_L) or not os.path.exists(mask_R):
            return {
                "patient_id": patient_id,
                "status": "mask_missing",
                "class_name": class_name,
                "label": label,
                "error": "mask_L ou mask_R absent"
            }

        # 4) Déplacer sorties HippoDeep
        patient_hippo_out_dir = move_hippodeep_outputs(
            patient_id,
            mask_L,
            mask_R,
            brain_mask,
            vol_csv,
            hippo_mask_dir
        )

        # 5) Charger image originale + masques
        mri_data, _ = load_nifti(raw_path)

        moved_mask_L = os.path.join(patient_hippo_out_dir, f"{patient_id}_mask_L.nii.gz")
        moved_mask_R = os.path.join(patient_hippo_out_dir, f"{patient_id}_mask_R.nii.gz")

        union_mask = get_union_mask(moved_mask_L, moved_mask_R)

        if mri_data.shape != union_mask.shape:
            return {
                "patient_id": patient_id,
                "status": "shape_mismatch",
                "class_name": class_name,
                "label": label,
                "error": f"MRI={mri_data.shape}, MASK={union_mask.shape}"
            }

        # 6) Visualisations IRM originale + masque + overlay
        save_mri_views(
            mri_data,
            original_qc_path,
            title=f"{patient_id} | {class_name} | IRM originale"
        )

        save_mask_views(
            union_mask.astype(np.float32),
            mask_qc_path,
            title=f"{patient_id} | {class_name} | Masque hippocampe L+R"
        )

        save_overlay_views(
            mri_data,
            union_mask.astype(np.float32),
            overlay_qc_path,
            title=f"{patient_id} | {class_name} | IRM + hippocampe"
        )

        # 7) Bounding box autour hippocampe gauche + droit
        bbox = get_bounding_box(union_mask)

        if bbox is None:
            return {
                "patient_id": patient_id,
                "status": "empty_mask",
                "class_name": class_name,
                "label": label,
                "error": "masque vide"
            }

        bbox_expanded = expand_bounding_box(
            bbox,
            mri_data.shape,
            MARGIN
        )

        # 8) Crop ROI
        roi = crop_volume(mri_data, bbox_expanded)

        # 9) Resize vers 64x64x64
        roi_resized = resize_volume(roi, TARGET_SHAPE)

        # 10) Normalisation
        roi_norm = normalize_volume(roi_resized, use_zscore=USE_ZSCORE)

        # 11) Sauvegarde ROI
        np.save(roi_output_path, roi_norm)

        # 12) Visualisation ROI 2D
        save_roi_views(
            roi_norm,
            roi_qc_path,
            title=f"{patient_id} | {class_name} | ROI hippocampe 64x64x64"
        )

        # 13) Visualisation ROI 3D
        if SAVE_3D_VIEW:
            save_roi_3d_view(
                roi_norm,
                roi_3d_qc_path,
                title=f"{patient_id} | {class_name} | ROI hippocampe 3D"
            )

        return {
            "patient_id": patient_id,
            "status": "ok",
            "class_name": class_name,
            "label": label,
            "roi_path": roi_output_path,
            "mask_dir": patient_hippo_out_dir,
            "bbox_raw": str(bbox),
            "bbox_expanded": str(bbox_expanded),
            "roi_shape_before_resize": str(roi.shape),
            "roi_shape_final": str(roi_norm.shape),
            "min": float(roi_norm.min()),
            "max": float(roi_norm.max()),
            "mean": float(roi_norm.mean()),
            "std": float(roi_norm.std()),
            "qc_original_path": original_qc_path,
            "qc_mask_path": mask_qc_path,
            "qc_overlay_path": overlay_qc_path,
            "qc_roi_path": roi_qc_path,
            "qc_roi_3d_path": roi_3d_qc_path if SAVE_3D_VIEW else None,
        }

    except Exception as e:
        return {
            "patient_id": patient_id,
            "status": "error",
            "class_name": class_name,
            "label": label,
            "error": str(e)
        }

    finally:
        # Supprimer le fichier temporaire
        if os.path.exists(working_nii):
            os.remove(working_nii)


In [ ]:
# =========================================================
# 5) TRAITEMENT PAR CLASSE
# =========================================================

def process_class_dataset(class_name):
    input_root = CLASS_INPUTS[class_name]
    label = CLASS_LABELS[class_name]

    work_root = os.path.join(OUTPUT_ROOT, class_name)
    hippo_mask_dir = os.path.join(work_root, "hippodeep_outputs")
    roi_output_dir = os.path.join(work_root, "roi_npy", class_name)

    safe_mkdir(work_root)
    safe_mkdir(hippo_mask_dir)
    safe_mkdir(roi_output_dir)

    files = list_mri_files(input_root)

    print("\n" + "=" * 80)
    print(f"Pipeline HippoDeep + ROI | {class_name}")
    print("=" * 80)
    print("INPUT_ROOT  :", input_root)
    print("Nb fichiers :", len(files))
    print("WORK_ROOT   :", work_root)

    results = []

    for filename in tqdm(files, desc=f"Traitement {class_name}"):
        result = process_one_patient(
            filename=filename,
            class_name=class_name,
            input_root=input_root,
            hippo_mask_dir=hippo_mask_dir,
            roi_output_dir=roi_output_dir,
            label=label
        )

        results.append(result)

    df = pd.DataFrame(results)

    metadata_csv = os.path.join(work_root, f"metadata_{class_name}_{DATASET_NAME}.csv")
    df.to_csv(metadata_csv, index=False)

    print("\n----- Résumé", class_name, "-----")
    print(df["status"].value_counts(dropna=False))
    print("Metadata :", metadata_csv)

    ok_count = (df["status"] == "ok").sum()
    skipped_count = (df["status"] == "skipped_existing_roi").sum()
    print("Nb ROI réussies :", ok_count)
    print("Nb ROI déjà existantes :", skipped_count)
    print("ROI dir :", roi_output_dir)

    return df, roi_output_dir, metadata_csv


In [ ]:
# =========================================================
# 6) LANCEMENT
# =========================================================

df_ad, roi_ad_dir, meta_ad = process_class_dataset("AD")
df_cn, roi_cn_dir, meta_cn = process_class_dataset("CN")

print("\n" + "=" * 80)
print("PIPELINE TERMINÉ")
print("=" * 80)

print("AD metadata :", meta_ad)
print("CN metadata :", meta_cn)

print("AD ROI dir :", roi_ad_dir)
print("CN ROI dir :", roi_cn_dir)


In [ ]:
# =========================================================
# 7) MÉTADONNÉES GLOBALES
# =========================================================

df_all = pd.concat([df_ad, df_cn], axis=0).reset_index(drop=True)

global_metadata_csv = os.path.join(OUTPUT_ROOT, f"metadata_all_{DATASET_NAME}.csv")
df_all.to_csv(global_metadata_csv, index=False)

print("\nMetadata globale :", global_metadata_csv)
print("\nStatus global :")
print(df_all["status"].value_counts(dropna=False))

print("\nDistribution classes :")
print(df_all["class_name"].value_counts(dropna=False))


In [ ]:
# =========================================================
# 8) AFFICHAGE DE QUELQUES RÉSULTATS
# =========================================================

show_images_from_folder(QC_ORIGINAL_DIR, n=N_SHOW, title="IRM originales en noir et blanc")
show_images_from_folder(QC_MASK_DIR, n=N_SHOW, title="Masques hippocampe L+R")
show_images_from_folder(QC_OVERLAY_DIR, n=N_SHOW, title="Overlay IRM + hippocampe")
show_images_from_folder(QC_ROI_DIR, n=N_SHOW, title="ROI hippocampe 2D")

if SAVE_3D_VIEW:
    show_images_from_folder(QC_ROI_3D_DIR, n=N_SHOW, title="ROI hippocampe 3D")


In [ ]:
# =========================================================
# 9) ZIP DES RÉSULTATS VISUELS + ROI
# =========================================================

ZIP_PATH = f"/kaggle/working/hippodeep_roi_{DATASET_NAME}_with_visualisations.zip"

shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=OUTPUT_ROOT
)

print("\nZIP créé :", ZIP_PATH)
print("Existe :", os.path.exists(ZIP_PATH))


## 3. Poids pré-entraînés MedicalNet

Cellule issue du notebook original pour récupérer les poids nécessaires.
Le modèle final utilise **ResNet34**.


In [ ]:
# =========================================================
# TÉLÉCHARGER LES POIDS MANQUANTS MEDICALNET
# =========================================================

from pathlib import Path
import os

WEIGHTS_DIR = Path("/kaggle/working/medicalnet_weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

downloads = {
    "resnet10": {
        "path": WEIGHTS_DIR / "resnet_10_23dataset.pth",
        "url": "https://huggingface.co/TencentMedicalNet/MedicalNet-Resnet10/resolve/main/resnet_10_23dataset.pth"
    },
    "resnet34": {
        "path": WEIGHTS_DIR / "resnet_34_23dataset.pth",
        "url": "https://huggingface.co/TencentMedicalNet/MedicalNet-Resnet34/resolve/main/resnet_34_23dataset.pth"
    },
    "resnet50": {
        "path": WEIGHTS_DIR / "resnet_50_23dataset.pth",
        "url": "https://huggingface.co/TencentMedicalNet/MedicalNet-Resnet50/resolve/main/resnet_50_23dataset.pth"
    },
}

for name, info in downloads.items():
    path = info["path"]
    url = info["url"]

    if path.exists():
        print(name, "déjà téléchargé :", path)
    else:
        print("Téléchargement :", name)
        os.system(f'wget -O "{path}" "{url}"')

    print(name, "exists:", path.exists())
    if path.exists():
        print("taille MB:", path.stat().st_size / (1024 * 1024))

## 4. Modèle final hybride

Le bloc suivant correspond au **pipeline final ResNet34 + features HippoDeep/ROI + BernoulliNB**.
Le sommaire commenté au début du bloc reste avec les imports, puis le code est découpé à partir des vraies sections d'implémentation.


In [ ]:
# =========================================================
# MODÈLE FINAL HYBRIDE COMPLET
# CNN 3D MedicalNet/MONAI ResNet34 + Features HippoDeep/ROI + BernoulliNB
#
# Pipeline :
# 1. Charger ROI AD/CN
# 2. Construire dataset final : 133 AD + 134 CN = 267 patients
# 3. Split train/val/test
# 4. Entraîner CNN ResNet34 avec data augmentation ultra-légère
# 5. Évaluer CNN seul avec seuil 0.61
# 6. Extraire cnn_prob_AD + cnn_pred_061
# 7. Extraire features anatomiques/statistiques
# 8. Entraîner BernoulliNB final
# 9. Générer courbes, ROC, matrices, tableaux, ZIP final
# =========================================================

import os
import re
import json
import random
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import nibabel as nib
from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, Binarizer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

# MONAI ResNet34 3D
try:
    from monai.networks.nets import resnet34
except Exception as e:
    raise ImportError("MONAI n'est pas installé. Installe-le avec : pip install monai") from e


In [ ]:
# =========================================================
# 1. CONFIGURATION
# =========================================================

SEED = 42
SPLIT_SEED = 2026

ROI_ROOT = Path("/kaggle/working/hippodeep_roi_FINAL_SC_1Y_15T")
AD_ROI_DIR = ROI_ROOT / "AD" / "roi_npy" / "AD"
CN_ROI_DIR = ROI_ROOT / "CN" / "roi_npy" / "CN"

HIPPODEEP_ROOT = ROI_ROOT

# Si ton fichier pré-entraîné est ailleurs, indique son chemin ici.
# Sinon, le code essaie de le trouver automatiquement.
PRETRAINED_PATH = None

OUTPUT_DIR = Path("/kaggle/working/FINAL_HYBRID_RESNET34_BERNOULLINB_RERUN")
CNN_DIR = OUTPUT_DIR / "cnn"
ML_DIR = OUTPUT_DIR / "ml"
FEATURE_DIR = OUTPUT_DIR / "features"
FIG_DIR = OUTPUT_DIR / "figures"
SPLIT_DIR = OUTPUT_DIR / "split"

for d in [OUTPUT_DIR, CNN_DIR, ML_DIR, FEATURE_DIR, FIG_DIR, SPLIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BEST_CNN_PATH = CNN_DIR / "best_resnet34_cnn.pth"
BEST_ML_PATH = ML_DIR / "bernoulliNB_hybrid_model.joblib"
FEATURE_COLUMNS_PATH = ML_DIR / "feature_columns.txt"
BEST_THRESHOLD_PATH = ML_DIR / "best_hybrid_threshold.txt"

CNN_THRESHOLD = 0.61
K_FEATURES = 12
BERNOULLI_ALPHA = 0.5
MIN_RECALL_AD = 0.80

DROPOUT = 0.55
EPOCHS_HEAD = 10
EPOCHS_FINE = 40
LR_HEAD = 1e-3
LR_FINE = 5e-6
MIN_LR_HEAD = 1e-5
MIN_LR_FINE = 1e-8
PATIENCE_HEAD = 5
PATIENCE_FINE = 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_MULTI_GPU = torch.cuda.device_count() > 1
BATCH_SIZE = 8 if USE_MULTI_GPU else 4
NUM_WORKERS = 2

print("DEVICE :", DEVICE)
print("GPU count :", torch.cuda.device_count())
print("BATCH_SIZE :", BATCH_SIZE)
print("OUTPUT_DIR :", OUTPUT_DIR)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


In [ ]:
# =========================================================
# 2. CONSTRUIRE DATASET FINAL 133 AD + 134 CN
# =========================================================

def extract_subject_id(text):
    text = str(text)
    m = re.search(r"\d{3}_S_\d{4}", text)
    if m:
        return m.group(0)
    name = Path(text).name
    name = name.replace(".npy", "").replace("_roi", "")
    return name


def build_dataset_267(ad_dir, cn_dir):
    ad_files = sorted(list(Path(ad_dir).glob("*.npy")))
    cn_files = sorted(list(Path(cn_dir).glob("*.npy")))

    print("ROI AD trouvées :", len(ad_files))
    print("ROI CN trouvées :", len(cn_files))

    if len(ad_files) < 133:
        raise ValueError("Moins de 133 ROI AD trouvées.")
    if len(cn_files) < 134:
        raise ValueError("Moins de 134 ROI CN trouvées.")

    # Tous les AD disponibles = 133
    selected_ad = ad_files[:133]

    # Sélection reproductible de 134 CN
    rng = np.random.default_rng(SPLIT_SEED)
    selected_cn = list(rng.choice(cn_files, size=134, replace=False))

    rows = []

    for p in selected_ad:
        rows.append({
            "subject_id": extract_subject_id(p),
            "roi_path": str(p),
            "class_name": "AD",
            "label": 1
        })

    for p in selected_cn:
        rows.append({
            "subject_id": extract_subject_id(p),
            "roi_path": str(p),
            "class_name": "CN",
            "label": 0
        })

    df = pd.DataFrame(rows)
    df = df.sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)

    return df


full_df = build_dataset_267(AD_ROI_DIR, CN_ROI_DIR)

print("\nDataset final utilisé :")
print(full_df["class_name"].value_counts())
print("Total :", len(full_df))

full_df.to_csv(SPLIT_DIR / "full_dataset_267.csv", index=False)


In [ ]:
# =========================================================
# 3. SPLIT TRAIN / VALIDATION / TEST
# =========================================================

train_df, temp_df = train_test_split(
    full_df,
    test_size=0.30,
    stratify=full_df["label"],
    random_state=SPLIT_SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SPLIT_SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
val_df.to_csv(SPLIT_DIR / "val.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

print("\nSplit final :")
print("Train :")
print(train_df["class_name"].value_counts())
print("Val :")
print(val_df["class_name"].value_counts())
print("Test :")
print(test_df["class_name"].value_counts())


In [ ]:
# =========================================================
# 4. DATASET PYTORCH + AUGMENTATION ULTRA-LÉGÈRE
# =========================================================

def normalize_volume_np(x):
    x = np.nan_to_num(x).astype(np.float32)
    mean = x.mean()
    std = x.std()

    if std < 1e-8:
        return np.zeros_like(x, dtype=np.float32)

    return ((x - mean) / std).astype(np.float32)


def ultra_light_intensity(x, p=0.10):
    if np.random.rand() < p:
        scale = np.random.uniform(0.98, 1.02)
        shift = np.random.uniform(-0.02, 0.02)
        x = x * scale + shift
    return x.astype(np.float32)


def ultra_light_noise(x, p=0.10):
    if np.random.rand() < p:
        noise = np.random.normal(0, 0.005, size=x.shape).astype(np.float32)
        x = x + noise
    return x.astype(np.float32)


def augment_ultra_light(x):
    x = ultra_light_intensity(x, p=0.10)
    x = ultra_light_noise(x, p=0.10)
    return x.astype(np.float32)


class RoiDataset(Dataset):
    def __init__(self, dataframe, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        x = np.load(row["roi_path"]).astype(np.float32)
        y = np.float32(row["label"])

        x = normalize_volume_np(x)

        if self.augment:
            x = augment_ultra_light(x)
            x = normalize_volume_np(x)

        x = np.expand_dims(x, axis=0)  # (1, 64, 64, 64)

        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


train_loader = DataLoader(
    RoiDataset(train_df, augment=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    RoiDataset(val_df, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    RoiDataset(test_df, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


In [ ]:
# =========================================================
# 5. TROUVER / CHARGER POIDS RESNET34 PRÉ-ENTRAÎNÉS
# =========================================================

def find_pretrained_resnet34():
    patterns = [
        "*resnet_34*.pth",
        "*resnet34*.pth",
        "*resnet_34*.pt",
        "*resnet34*.pt"
    ]

    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    found = []

    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            found.extend(list(root.rglob(pattern)))

    # éviter les modèles finaux déjà entraînés
    filtered = []
    for p in found:
        s = str(p).lower()
        if "best_" in s or "final" in s or "test" in s or "hybrid" in s:
            continue
        filtered.append(p)

    found = filtered if len(filtered) > 0 else found
    found = sorted(found, key=lambda p: len(str(p)))

    if len(found) == 0:
        raise FileNotFoundError(
            "Aucun poids ResNet34 trouvé. Ajoute resnet_34*.pth dans /kaggle/input."
        )

    print("\nPoids pré-entraînés trouvés :")
    for p in found[:10]:
        print("-", p)

    return found[0]


if PRETRAINED_PATH is None:
    PRETRAINED_PATH = find_pretrained_resnet34()
else:
    PRETRAINED_PATH = Path(PRETRAINED_PATH)

print("\nPRETRAINED_PATH utilisé :", PRETRAINED_PATH)


In [ ]:
# =========================================================
# 6. CONSTRUIRE RESNET34 3D
# =========================================================

def build_resnet34_model(pretrained_path, dropout=0.55):
    model = resnet34(
        spatial_dims=3,
        n_input_channels=1,
        num_classes=1,
        feed_forward=True,
        pretrained=False
    )

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 1)
    )

    checkpoint = torch.load(
        pretrained_path,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "net" in checkpoint:
            state_dict = checkpoint["net"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    model_state = model.state_dict()
    cleaned_state = {}

    for k, v in state_dict.items():
        new_k = str(k).replace("module.", "").replace("model.", "")

        if new_k.startswith("fc."):
            continue

        if new_k in model_state and tuple(model_state[new_k].shape) == tuple(v.shape):
            cleaned_state[new_k] = v

    missing, unexpected = model.load_state_dict(cleaned_state, strict=False)

    print("\nChargement poids pré-entraînés :")
    print("Poids compatibles chargés :", len(cleaned_state))
    print("Missing keys :", len(missing))
    print("Unexpected keys :", len(unexpected))

    if len(cleaned_state) == 0:
        raise ValueError(
            "Aucun poids pré-entraîné compatible n'a été chargé. "
            "Vérifie que le fichier correspond bien à ResNet34 3D MONAI/MedicalNet."
        )

    return model


model = build_resnet34_model(PRETRAINED_PATH, dropout=DROPOUT)

if USE_MULTI_GPU:
    model = nn.DataParallel(model)

model = model.to(DEVICE)


In [ ]:
# =========================================================
# 7. FREEZE / UNFREEZE
# =========================================================

def raw_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def freeze_backbone(model):
    m = raw_model(model)

    for p in m.parameters():
        p.requires_grad = False

    for p in m.fc.parameters():
        p.requires_grad = True

    print("Backbone gelé : seule fc est entraînée.")


def unfreeze_layer4(model):
    m = raw_model(model)

    for p in m.parameters():
        p.requires_grad = False

    for p in m.layer4.parameters():
        p.requires_grad = True

    for p in m.fc.parameters():
        p.requires_grad = True

    print("Fine-tuning : layer4 + fc entraînés.")


def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Paramètres totaux :", total)
    print("Paramètres entraînables :", trainable)


def save_checkpoint(model, path, extra_info=None):
    m = raw_model(model)
    torch.save({
        "model_state_dict": m.state_dict(),
        "extra_info": extra_info or {}
    }, path)


def load_checkpoint(model, path):
    m = raw_model(model)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt["model_state_dict"])
    return model, ckpt


In [ ]:
# =========================================================
# 8. TRAINING CNN
# =========================================================

def set_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group["lr"] = lr


def warmup_cosine_lr(epoch, total_epochs, base_lr, min_lr, warmup_epochs):
    if epoch <= warmup_epochs:
        return base_lr * epoch / warmup_epochs

    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    cosine = 0.5 * (1 + np.cos(np.pi * progress))

    return min_lr + (base_lr - min_lr) * cosine


def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true = []
    all_prob = []

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(x).view(-1)
            loss = criterion(logits, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
                optimizer.step()

            probs = torch.sigmoid(logits).detach().cpu().numpy()

            total_loss += loss.item()
            all_true.extend(y.detach().cpu().numpy().astype(int).tolist())
            all_prob.extend(probs.tolist())

    all_true = np.array(all_true)
    all_prob = np.array(all_prob)
    all_pred = (all_prob >= 0.5).astype(int)

    try:
        auc = roc_auc_score(all_true, all_prob)
    except Exception:
        auc = np.nan

    return {
        "loss": total_loss / max(1, len(loader)),
        "accuracy": accuracy_score(all_true, all_pred),
        "precision": precision_score(all_true, all_pred, zero_division=0),
        "recall": recall_score(all_true, all_pred, zero_division=0),
        "f1": f1_score(all_true, all_pred, zero_division=0),
        "auc": auc
    }


def train_phase(model, train_loader, val_loader, epochs, base_lr, min_lr,
                warmup_epochs, save_path, phase_name, patience):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=base_lr,
        weight_decay=1e-4
    )

    best_auc = -1.0
    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        lr = warmup_cosine_lr(
            epoch=epoch,
            total_epochs=epochs,
            base_lr=base_lr,
            min_lr=min_lr,
            warmup_epochs=warmup_epochs
        )

        set_lr(optimizer, lr)

        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None)

        val_auc = val_metrics["auc"]
        if np.isnan(val_auc):
            val_auc = 0.0

        row = {
            "phase": phase_name,
            "epoch": epoch,
            "lr": lr,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }

        history.append(row)

        print(f"\n[{phase_name}] Epoch {epoch}/{epochs} | LR={lr:.8f}")
        print(
            f"Train | loss={train_metrics['loss']:.4f} "
            f"acc={train_metrics['accuracy']:.4f} "
            f"auc={train_metrics['auc']:.4f} "
            f"f1={train_metrics['f1']:.4f}"
        )
        print(
            f"Val   | loss={val_metrics['loss']:.4f} "
            f"acc={val_metrics['accuracy']:.4f} "
            f"auc={val_metrics['auc']:.4f} "
            f"f1={val_metrics['f1']:.4f} "
            f"recall={val_metrics['recall']:.4f}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0
            save_checkpoint(
                model,
                save_path,
                extra_info={
                    "best_val_auc": float(best_auc),
                    "phase": phase_name,
                    "epoch": epoch,
                    "augmentation": "ultra_light_intensity_noise"
                }
            )
            print("Nouveau meilleur CNN sauvegardé.")
        else:
            patience_counter += 1
            print(f"Patience : {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping.")
            break

    return model, pd.DataFrame(history)


print("\n================ PHASE 1 : FC seulement ================")
freeze_backbone(model)
count_trainable_params(model)

model, hist_head = train_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS_HEAD,
    base_lr=LR_HEAD,
    min_lr=MIN_LR_HEAD,
    warmup_epochs=3,
    save_path=BEST_CNN_PATH,
    phase_name="phase1_fc",
    patience=PATIENCE_HEAD
)

hist_head.to_csv(CNN_DIR / "history_head.csv", index=False)

print("\n================ PHASE 2 : layer4 + FC ================")
model, ckpt = load_checkpoint(model, BEST_CNN_PATH)

unfreeze_layer4(model)
count_trainable_params(model)

model, hist_fine = train_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS_FINE,
    base_lr=LR_FINE,
    min_lr=MIN_LR_FINE,
    warmup_epochs=4,
    save_path=BEST_CNN_PATH,
    phase_name="phase2_layer4_fc",
    patience=PATIENCE_FINE
)

hist_fine.to_csv(CNN_DIR / "history_fine.csv", index=False)

history_all = pd.concat([hist_head, hist_fine], axis=0).reset_index(drop=True)
history_all.to_csv(CNN_DIR / "history_all.csv", index=False)

model, ckpt = load_checkpoint(model, BEST_CNN_PATH)
model.eval()

print("\nMeilleur CNN chargé :")
print(ckpt.get("extra_info", {}))


In [ ]:
# =========================================================
# 9. PRÉDICTIONS CNN
# =========================================================

def get_predictions(model, loader):
    model.eval()
    y_true = []
    y_prob = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            logits = model(x).view(-1)
            probs = torch.sigmoid(logits).detach().cpu().numpy()

            y_prob.extend(probs.tolist())
            y_true.extend(y.numpy().astype(int).tolist())

    return np.array(y_true), np.array(y_prob)


def evaluate_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "precision_AD": precision_score(y_true, y_pred, zero_division=0),
        "recall_AD": recall_score(y_true, y_pred, zero_division=0),
        "f1_AD": f1_score(y_true, y_pred, zero_division=0),
        "specificity_CN": tn / (tn + fp + 1e-8),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }, cm, y_pred


def search_best_threshold(y_true, y_prob, min_recall_ad=None):
    rows = []

    for th in np.arange(0.05, 0.96, 0.01):
        metrics, _, _ = evaluate_threshold(y_true, y_prob, th)
        rows.append(metrics)

    df = pd.DataFrame(rows)

    if min_recall_ad is not None:
        candidates = df[df["recall_AD"] >= min_recall_ad].copy()

        if len(candidates) > 0:
            best_row = candidates.sort_values(
                ["accuracy", "f1_AD", "specificity_CN"],
                ascending=False
            ).iloc[0]
        else:
            best_row = df.sort_values(
                ["accuracy", "f1_AD", "recall_AD"],
                ascending=False
            ).iloc[0]
    else:
        best_row = df.sort_values(
            ["accuracy", "f1_AD", "recall_AD"],
            ascending=False
        ).iloc[0]

    return float(best_row["threshold"]), df, best_row


y_train_true, y_train_prob = get_predictions(model, train_loader)
y_val_true, y_val_prob = get_predictions(model, val_loader)
y_test_true, y_test_prob = get_predictions(model, test_loader)

cnn_metrics_061, cm_cnn_061, pred_cnn_061 = evaluate_threshold(
    y_test_true,
    y_test_prob,
    CNN_THRESHOLD
)

print("\nRésultat CNN seuil 0.61 :")
display(pd.DataFrame([cnn_metrics_061]))

print("\nClassification report CNN seuil 0.61 :")
print(classification_report(
    y_test_true,
    pred_cnn_061,
    target_names=["CN", "AD"],
    digits=4
))

print("Matrice CNN seuil 0.61 :")
print(cm_cnn_061)

pd.DataFrame([cnn_metrics_061]).to_csv(CNN_DIR / "cnn_metrics_threshold061.csv", index=False)

# Sauvegarde prédictions CNN
for split_name, df_split, probs in [
    ("train", train_df, y_train_prob),
    ("val", val_df, y_val_prob),
    ("test", test_df, y_test_prob)
]:
    out = df_split.copy()
    out["cnn_prob_AD"] = probs
    out["cnn_pred_061"] = (probs >= CNN_THRESHOLD).astype(int)
    out.to_csv(FEATURE_DIR / f"{split_name}_cnn_predictions.csv", index=False)


In [ ]:
# =========================================================
# 10. COURBES CNN
# =========================================================

def save_training_curves(hist, fig_dir):
    plt.figure(figsize=(9, 5))
    plt.plot(hist.index, hist["train_loss"], label="Train loss")
    plt.plot(hist.index, hist["val_loss"], label="Validation loss")
    plt.title("CNN ResNet34 - Loss")
    plt.xlabel("Étapes")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fig_dir / "cnn_curve_loss.png", dpi=150)
    plt.show()

    plt.figure(figsize=(9, 5))
    plt.plot(hist.index, hist["train_accuracy"], label="Train accuracy")
    plt.plot(hist.index, hist["val_accuracy"], label="Validation accuracy")
    plt.title("CNN ResNet34 - Accuracy")
    plt.xlabel("Étapes")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fig_dir / "cnn_curve_accuracy.png", dpi=150)
    plt.show()

    plt.figure(figsize=(9, 5))
    plt.plot(hist.index, hist["train_auc"], label="Train AUC")
    plt.plot(hist.index, hist["val_auc"], label="Validation AUC")
    plt.title("CNN ResNet34 - AUC")
    plt.xlabel("Étapes")
    plt.ylabel("AUC")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fig_dir / "cnn_curve_auc.png", dpi=150)
    plt.show()

    plt.figure(figsize=(9, 5))
    plt.plot(hist.index, hist["train_f1"], label="Train F1")
    plt.plot(hist.index, hist["val_f1"], label="Validation F1")
    plt.title("CNN ResNet34 - F1-score")
    plt.xlabel("Étapes")
    plt.ylabel("F1-score")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fig_dir / "cnn_curve_f1.png", dpi=150)
    plt.show()

    plt.figure(figsize=(9, 5))
    plt.plot(hist.index, hist["lr"], label="Learning rate")
    plt.title("CNN ResNet34 - Learning rate")
    plt.xlabel("Étapes")
    plt.ylabel("Learning rate")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fig_dir / "cnn_curve_learning_rate.png", dpi=150)
    plt.show()


save_training_curves(history_all, FIG_DIR)

fpr, tpr, _ = roc_curve(y_test_true, y_test_prob)
auc_cnn = roc_auc_score(y_test_true, y_test_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"CNN AUC={auc_cnn:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("CNN ResNet34 - ROC test")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "cnn_roc_test.png", dpi=150)
plt.show()

plt.figure(figsize=(5, 4))
plt.imshow(cm_cnn_061)
plt.title("CNN - Matrice de confusion seuil 0.61")
plt.xticks([0, 1], ["CN", "AD"])
plt.yticks([0, 1], ["CN", "AD"])
plt.xlabel("Prédit")
plt.ylabel("Réel")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_cnn_061[i, j], ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR / "cnn_confusion_matrix_threshold061.png", dpi=150)
plt.show()


In [ ]:
# =========================================================
# 11. EXTRACTION FEATURES HIPPODEEP + ROI
# =========================================================

def load_nifti_data(path):
    img = nib.load(str(path))
    data = img.get_fdata().astype(np.float32)
    data = np.nan_to_num(data)
    return img, data


def compute_volume_mm3(mask, voxel_sizes):
    voxel_volume = float(voxel_sizes[0] * voxel_sizes[1] * voxel_sizes[2])
    return float(mask.sum() * voxel_volume)


def safe_ratio(a, b):
    if b is None or np.isnan(b) or b <= 1e-8:
        return np.nan
    return float(a / b)


def find_mask_file(subject_id, class_name, side):
    class_root = HIPPODEEP_ROOT / class_name

    if side == "L":
        patterns = [f"*{subject_id}*mask_L*.nii*", f"*{subject_id}*_L*.nii*"]
    else:
        patterns = [f"*{subject_id}*mask_R*.nii*", f"*{subject_id}*_R*.nii*"]

    roots = [class_root, Path("/kaggle/working")]

    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            files = sorted(list(root.rglob(pattern)))
            files = [f for f in files if "brain" not in f.name.lower()]
            if len(files) > 0:
                return files[0]

    return None


def find_brain_mask(subject_id, class_name):
    class_root = HIPPODEEP_ROOT / class_name
    roots = [class_root, Path("/kaggle/working")]

    patterns = [
        f"*{subject_id}*brain_mask*.nii*",
        f"*{subject_id}*brain*.nii*"
    ]

    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            files = sorted(list(root.rglob(pattern)))
            if len(files) > 0:
                return files[0]

    return None


def compute_roi_stats(roi_path):
    roi = np.load(roi_path).astype(np.float32)
    roi = np.nan_to_num(roi)
    flat = roi.reshape(-1)

    d = {}
    d["roi_mean"] = float(np.mean(flat))
    d["roi_std"] = float(np.std(flat))
    d["roi_median"] = float(np.median(flat))
    d["roi_min"] = float(np.min(flat))
    d["roi_max"] = float(np.max(flat))
    d["roi_p01"] = float(np.percentile(flat, 1))
    d["roi_p05"] = float(np.percentile(flat, 5))
    d["roi_p10"] = float(np.percentile(flat, 10))
    d["roi_p25"] = float(np.percentile(flat, 25))
    d["roi_p75"] = float(np.percentile(flat, 75))
    d["roi_p90"] = float(np.percentile(flat, 90))
    d["roi_p95"] = float(np.percentile(flat, 95))
    d["roi_p99"] = float(np.percentile(flat, 99))
    d["roi_iqr"] = d["roi_p75"] - d["roi_p25"]
    d["roi_range"] = d["roi_max"] - d["roi_min"]
    d["roi_energy"] = float(np.mean(flat ** 2))

    if np.std(flat) > 1e-8:
        d["roi_skew"] = float(skew(flat))
        d["roi_kurtosis"] = float(kurtosis(flat))
    else:
        d["roi_skew"] = 0.0
        d["roi_kurtosis"] = 0.0

    return d


def compute_hippo_features(subject_id, class_name):
    mask_L_path = find_mask_file(subject_id, class_name, "L")
    mask_R_path = find_mask_file(subject_id, class_name, "R")
    brain_mask_path = find_brain_mask(subject_id, class_name)

    d = {
        "has_mask_L": int(mask_L_path is not None),
        "has_mask_R": int(mask_R_path is not None),
        "has_brain_mask": int(brain_mask_path is not None),
        "mask_L_path": str(mask_L_path) if mask_L_path is not None else None,
        "mask_R_path": str(mask_R_path) if mask_R_path is not None else None,
        "brain_mask_path": str(brain_mask_path) if brain_mask_path is not None else None,
    }

    vol_L = np.nan
    vol_R = np.nan
    vol_total = np.nan
    vol_brain = np.nan

    if mask_L_path is not None:
        img_L, data_L = load_nifti_data(mask_L_path)
        mask_L = (data_L > 0).astype(np.uint8)
        zooms = img_L.header.get_zooms()[:3]
        vol_L = compute_volume_mm3(mask_L, zooms)
        d["voxels_L"] = int(mask_L.sum())
    else:
        d["voxels_L"] = np.nan

    if mask_R_path is not None:
        img_R, data_R = load_nifti_data(mask_R_path)
        mask_R = (data_R > 0).astype(np.uint8)
        zooms = img_R.header.get_zooms()[:3]
        vol_R = compute_volume_mm3(mask_R, zooms)
        d["voxels_R"] = int(mask_R.sum())
    else:
        d["voxels_R"] = np.nan

    if not np.isnan(vol_L) and not np.isnan(vol_R):
        vol_total = vol_L + vol_R

    if brain_mask_path is not None:
        img_B, data_B = load_nifti_data(brain_mask_path)
        brain = (data_B > 0).astype(np.uint8)
        zooms = img_B.header.get_zooms()[:3]
        vol_brain = compute_volume_mm3(brain, zooms)
        d["voxels_brain"] = int(brain.sum())
    else:
        d["voxels_brain"] = np.nan

    d["hippo_L_mm3"] = vol_L
    d["hippo_R_mm3"] = vol_R
    d["hippo_total_mm3"] = vol_total
    d["brain_volume_mm3"] = vol_brain

    d["hippo_L_over_brain"] = safe_ratio(vol_L, vol_brain)
    d["hippo_R_over_brain"] = safe_ratio(vol_R, vol_brain)
    d["hippo_total_over_brain"] = safe_ratio(vol_total, vol_brain)

    if not np.isnan(vol_L) and not np.isnan(vol_R) and (vol_L + vol_R) > 1e-8:
        d["hippo_asymmetry_abs"] = float(abs(vol_L - vol_R) / (vol_L + vol_R))
        d["hippo_L_minus_R"] = float(vol_L - vol_R)
        d["hippo_L_minus_R_over_total"] = float((vol_L - vol_R) / (vol_L + vol_R))
        d["hippo_L_over_R"] = safe_ratio(vol_L, vol_R)
        d["hippo_R_over_L"] = safe_ratio(vol_R, vol_L)
    else:
        d["hippo_asymmetry_abs"] = np.nan
        d["hippo_L_minus_R"] = np.nan
        d["hippo_L_minus_R_over_total"] = np.nan
        d["hippo_L_over_R"] = np.nan
        d["hippo_R_over_L"] = np.nan

    return d


def build_features_df(base_df, cnn_prob, split_name):
    rows = []

    base_df = base_df.reset_index(drop=True)

    for i, row in base_df.iterrows():
        subject_id = str(row["subject_id"])
        class_name = str(row["class_name"])
        roi_path = str(row["roi_path"])

        feat = {
            "subject_id": subject_id,
            "class_name": class_name,
            "label": int(row["label"]),
            "roi_path": roi_path,
            "cnn_prob_AD": float(cnn_prob[i]),
            "cnn_pred_061": int(cnn_prob[i] >= CNN_THRESHOLD)
        }

        feat.update(compute_roi_stats(roi_path))
        feat.update(compute_hippo_features(subject_id, class_name))

        rows.append(feat)

    out = pd.DataFrame(rows)
    out.to_csv(FEATURE_DIR / f"{split_name}_features_with_cnn.csv", index=False)

    print(f"\nFeatures {split_name} :", out.shape)
    print(out["class_name"].value_counts())

    return out


train_feat = build_features_df(train_df, y_train_prob, "train")
val_feat = build_features_df(val_df, y_val_prob, "val")
test_feat = build_features_df(test_df, y_test_prob, "test")


## 12. Benchmark des modèles ML avec LazyClassifier

Après l'extraction des caractéristiques anatomiques/statistiques et de la probabilité issue du CNN, plusieurs modèles de Machine Learning sont comparés afin d'identifier les approches les plus adaptées à la classification AD/CN.

Le benchmark est effectué sur l'ensemble de validation afin de conserver l'ensemble de test uniquement pour l'évaluation finale.

À l'issue de cette comparaison, **BernoulliNB** est retenu pour construire le modèle hybride final.

In [ ]:
# =========================================================
# 12. BENCHMARK ML AVEC LAZYCLASSIFIER
# =========================================================

import sys
import subprocess

# ---------------------------------------------------------
# 1. Installation / import LazyPredict
# ---------------------------------------------------------

try:
    from lazypredict.Supervised import LazyClassifier
    print("LazyPredict déjà installé.")
except ImportError:
    print("Installation de LazyPredict...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "lazypredict",
        "-q"
    ])
    from lazypredict.Supervised import LazyClassifier
    print("LazyPredict installé.")

# ---------------------------------------------------------
# 2. Préparer les features
# ---------------------------------------------------------

drop_cols_lazy = [
    "subject_id",
    "class_name",
    "label",
    "roi_path",
    "mask_L_path",
    "mask_R_path",
    "brain_mask_path",
    "has_mask_L",
    "has_mask_R",
    "has_brain_mask",
]

feature_cols_lazy = [
    c for c in train_feat.columns
    if c not in drop_cols_lazy
]

print("Nombre de features :", len(feature_cols_lazy))

# Copies pour ne pas modifier les DataFrames utilisés ensuite
train_lazy = train_feat.copy()
val_lazy = val_feat.copy()

# ---------------------------------------------------------
# 3. Nettoyage NaN / Inf
# ---------------------------------------------------------

for col in feature_cols_lazy:

    train_lazy[col] = train_lazy[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

    val_lazy[col] = val_lazy[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Médiane calculée uniquement sur TRAIN
    median_value = train_lazy[col].median()

    if np.isnan(median_value):
        median_value = 0.0

    train_lazy[col] = train_lazy[col].fillna(median_value)
    val_lazy[col] = val_lazy[col].fillna(median_value)

# ---------------------------------------------------------
# 4. Construire X / y
# ---------------------------------------------------------

X_train_lazy = train_lazy[feature_cols_lazy].values
y_train_lazy = train_lazy["label"].values

X_val_lazy = val_lazy[feature_cols_lazy].values
y_val_lazy = val_lazy["label"].values

print("X_train :", X_train_lazy.shape)
print("X_val   :", X_val_lazy.shape)

# ---------------------------------------------------------
# 5. LazyClassifier
# ---------------------------------------------------------

lazy_clf = LazyClassifier(
    verbose=0,
    ignore_warnings=True,
    custom_metric=None,
    predictions=False
)

lazy_results, _ = lazy_clf.fit(
    X_train_lazy,
    X_val_lazy,
    y_train_lazy,
    y_val_lazy
)

# ---------------------------------------------------------
# 6. Résultats
# ---------------------------------------------------------

print("\nTop modèles LazyClassifier :")
display(lazy_results.head(20))

# Sauvegarde
lazy_results.to_csv(
    ML_DIR / "lazyclassifier_validation_results.csv"
)

print(
    "\nRésultats sauvegardés dans :",
    ML_DIR / "lazyclassifier_validation_results.csv"
)

In [ ]:
# =========================================================
# 13. MODÈLE HYBRIDE BERNOULLINB
# =========================================================

drop_cols = [
    "subject_id",
    "class_name",
    "label",
    "roi_path",
    "mask_L_path",
    "mask_R_path",
    "brain_mask_path",
    "has_mask_L",
    "has_mask_R",
    "has_brain_mask",
]

feature_cols = [c for c in train_feat.columns if c not in drop_cols]

print("\nNombre de features disponibles :", len(feature_cols))
print(feature_cols)

# Imputation par médiane du train
for col in feature_cols:
    train_feat[col] = train_feat[col].replace([np.inf, -np.inf], np.nan)
    val_feat[col] = val_feat[col].replace([np.inf, -np.inf], np.nan)
    test_feat[col] = test_feat[col].replace([np.inf, -np.inf], np.nan)

    med = train_feat[col].median()
    if np.isnan(med):
        med = 0.0

    train_feat[col] = train_feat[col].fillna(med)
    val_feat[col] = val_feat[col].fillna(med)
    test_feat[col] = test_feat[col].fillna(med)

X_train = train_feat[feature_cols].values
y_train = train_feat["label"].values

X_val = val_feat[feature_cols].values
y_val = val_feat["label"].values

X_test = test_feat[feature_cols].values
y_test = test_feat["label"].values

K_REAL = min(K_FEATURES, len(feature_cols))

hybrid_model = Pipeline([
    ("scaler", StandardScaler()),
    ("binarizer", Binarizer(threshold=0.0)),
    ("select", SelectKBest(mutual_info_classif, k=K_REAL)),
    ("clf", BernoulliNB(alpha=BERNOULLI_ALPHA))
])

hybrid_model.fit(X_train, y_train)

val_prob_hybrid = hybrid_model.predict_proba(X_val)[:, 1]
test_prob_hybrid = hybrid_model.predict_proba(X_test)[:, 1]

best_hybrid_threshold, hybrid_threshold_df, best_hybrid_row = search_best_threshold(
    y_val,
    val_prob_hybrid,
    min_recall_ad=MIN_RECALL_AD
)

hybrid_threshold_df.to_csv(ML_DIR / "threshold_search_validation.csv", index=False)

with open(BEST_THRESHOLD_PATH, "w") as f:
    f.write(str(best_hybrid_threshold))

joblib.dump(hybrid_model, BEST_ML_PATH)

with open(FEATURE_COLUMNS_PATH, "w", encoding="utf-8") as f:
    for col in feature_cols:
        f.write(col + "\n")

hybrid_metrics, cm_hybrid, pred_hybrid = evaluate_threshold(
    y_test,
    test_prob_hybrid,
    best_hybrid_threshold
)

hybrid_metrics["model_name"] = "Hybrid_ResNet34_Features_BernoulliNB"
hybrid_metrics["n_features_total"] = len(feature_cols)
hybrid_metrics["n_features_selected"] = K_REAL

hybrid_metrics_df = pd.DataFrame([hybrid_metrics])
hybrid_metrics_df.to_csv(ML_DIR / "final_hybrid_test_metrics.csv", index=False)

print("\nRésultat final hybride :")
display(hybrid_metrics_df)

print("\nClassification report hybride :")
print(classification_report(
    y_test,
    pred_hybrid,
    target_names=["CN", "AD"],
    digits=4
))

print("Matrice de confusion hybride :")
print(cm_hybrid)

pred_df = test_feat[["subject_id", "class_name", "label", "roi_path"]].copy()
pred_df["cnn_prob_AD"] = test_feat["cnn_prob_AD"].values
pred_df["cnn_pred_061"] = test_feat["cnn_pred_061"].values
pred_df["hybrid_prob_AD"] = test_prob_hybrid
pred_df["hybrid_pred"] = pred_hybrid
pred_df["hybrid_pred_class"] = pred_df["hybrid_pred"].map({0: "CN", 1: "AD"})
pred_df.to_csv(ML_DIR / "test_predictions_hybrid.csv", index=False)

selector = hybrid_model.named_steps["select"]
selected_mask = selector.get_support()

selected_features = [
    f for f, keep in zip(feature_cols, selected_mask)
    if keep
]

selected_scores = selector.scores_[selected_mask]

selected_df = pd.DataFrame({
    "selected_feature": selected_features,
    "score": selected_scores
}).sort_values("score", ascending=False)

selected_df.to_csv(ML_DIR / "selected_features.csv", index=False)

print("\nFeatures sélectionnées :")
display(selected_df)


In [ ]:
# =========================================================
# 14. FIGURES HYBRIDES
# =========================================================

fpr, tpr, _ = roc_curve(y_test, test_prob_hybrid)
auc_hybrid = roc_auc_score(y_test, test_prob_hybrid)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Hybrid AUC={auc_hybrid:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Modèle hybride - ROC test")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "hybrid_roc_test.png", dpi=150)
plt.show()

plt.figure(figsize=(5, 4))
plt.imshow(cm_hybrid)
plt.title(f"Hybride - Matrice de confusion | seuil={best_hybrid_threshold:.2f}")
plt.xticks([0, 1], ["CN", "AD"])
plt.yticks([0, 1], ["CN", "AD"])
plt.xlabel("Prédit")
plt.ylabel("Réel")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_hybrid[i, j], ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR / "hybrid_confusion_matrix.png", dpi=150)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(hybrid_threshold_df["threshold"], hybrid_threshold_df["accuracy"], label="Val accuracy")
plt.plot(hybrid_threshold_df["threshold"], hybrid_threshold_df["recall_AD"], label="Val recall AD")
plt.plot(hybrid_threshold_df["threshold"], hybrid_threshold_df["specificity_CN"], label="Val specificity CN")
plt.axvline(best_hybrid_threshold, linestyle="--", label=f"Best threshold={best_hybrid_threshold:.2f}")
plt.title("Modèle hybride - choix du seuil sur validation")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "hybrid_threshold_validation.png", dpi=150)
plt.show()

comparison_df = pd.DataFrame([
    {"model": "CNN ResNet34 seuil 0.61", **cnn_metrics_061},
    {"model": "Hybride CNN + Features + BernoulliNB", **hybrid_metrics}
])

comparison_df.to_csv(OUTPUT_DIR / "comparison_cnn_vs_hybrid.csv", index=False)

print("\nComparaison CNN seul vs hybride :")
display(comparison_df[[
    "model",
    "threshold",
    "accuracy",
    "auc",
    "precision_AD",
    "recall_AD",
    "f1_AD",
    "specificity_CN",
    "tn",
    "fp",
    "fn",
    "tp"
]])

for metric in ["accuracy", "auc", "precision_AD", "recall_AD", "f1_AD", "specificity_CN"]:
    plt.figure(figsize=(8, 5))
    plt.bar(comparison_df["model"], comparison_df[metric])
    plt.title(f"Comparaison CNN vs Hybride - {metric}")
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"comparison_{metric}.png", dpi=150)
    plt.show()


In [ ]:
# =========================================================
# 15. README + MANIFEST + ZIP FINAL
# =========================================================

manifest = {
    "model_name": "FINAL_HYBRID_RESNET34_BERNOULLINB_RERUN",
    "cnn_checkpoint": str(BEST_CNN_PATH),
    "ml_model": str(BEST_ML_PATH),
    "feature_columns": str(FEATURE_COLUMNS_PATH),
    "best_hybrid_threshold": float(best_hybrid_threshold),
    "cnn_threshold": float(CNN_THRESHOLD),
    "features_selected": selected_features,
    "hybrid_metrics": {
        k: float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else str(v)
        for k, v in hybrid_metrics.items()
    }
}

with open(OUTPUT_DIR / "manifest_final_model.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / "README_FINAL_MODEL.txt", "w", encoding="utf-8") as f:
    f.write("MODÈLE FINAL HYBRIDE - RESNET34 + FEATURES + BERNOULLINB\n")
    f.write("=======================================================\n\n")
    f.write("Pipeline :\n")
    f.write("1. ROI hippocampique 64x64x64\n")
    f.write("2. Normalisation z-score\n")
    f.write("3. CNN MedicalNet/MONAI ResNet34\n")
    f.write("4. cnn_prob_AD + cnn_pred_061\n")
    f.write("5. Features volumes / ratios / asymétrie / statistiques ROI\n")
    f.write("6. StandardScaler + Binarizer + SelectKBest\n")
    f.write("7. BernoulliNB\n\n")

    f.write("Résultats CNN seuil 0.61 :\n")
    for k, v in cnn_metrics_061.items():
        f.write(f"{k}={v}\n")

    f.write("\nRésultats hybrides :\n")
    for k, v in hybrid_metrics.items():
        f.write(f"{k}={v}\n")

    f.write("\nFeatures sélectionnées :\n")
    for feat in selected_features:
        f.write(f"- {feat}\n")

FINAL_ZIP = "/kaggle/working/FINAL_HYBRID_RESNET34_BERNOULLINB_RERUN.zip"

shutil.make_archive(
    base_name=FINAL_ZIP.replace(".zip", ""),
    format="zip",
    root_dir=OUTPUT_DIR
)

print("\nZIP FINAL créé :")
print(FINAL_ZIP)
print("Existe :", os.path.exists(FINAL_ZIP))

print("\nDossier des figures :", FIG_DIR)
print("Modèle CNN :", BEST_CNN_PATH)
print("Modèle ML :", BEST_ML_PATH)
print("Seuil hybride :", BEST_THRESHOLD_PATH)


## 5. Résultat du run final enregistré dans le notebook original

Le run final sauvegardé dans le notebook source a produit :

| Métrique | Résultat |
|---|---:|
| Accuracy | **0.878049 ≈ 87.80 %** |
| ROC-AUC | **0.976190** |
| Precision AD | **0.826087** |
| Recall AD | **0.950000** |
| F1 AD | **0.883721** |
| Specificity CN | **0.809524** |
| Seuil hybride | **0.05** |

Matrice de confusion :

```text
[[17, 4],
 [ 1,19]]
```

Comparaison enregistrée dans le run final :
- CNN ResNet34 seul, seuil 0.61 : accuracy ≈ **58.54 %**, AUC ≈ **0.6524**
- Hybride CNN + features + BernoulliNB : accuracy ≈ **87.80 %**, AUC ≈ **0.9762**

Le ZIP final généré par le notebook original était :
`FINAL_HYBRID_RESNET34_BERNOULLINB_RERUN.zip`


## 6. Remarques de reproductibilité

- Le notebook est conçu pour l'environnement **Kaggle** utilisé pendant le projet.
- Les données IRM ne doivent pas être publiées dans le dépôt.
- Les poids ou artefacts lourds peuvent être récupérés à l'exécution ou stockés séparément.
- Les sorties ont été nettoyées pour GitHub, mais le code du pipeline final est conservé.
